In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\krish\anaconda3\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\krish\anaconda3\python.exe"

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
                    .appName("Age Group Segmentation of Air Passengers Using Spark RDDs") \
                    .master('local') \
                    .getOrCreate()

### A researcher wants to know the frequency of domestic air travel for various age groups. Let’s assume there are three age groups 18-30, 30-50 and 50 & above.

In [4]:
travel_rdd = spark.sparkContext.textFile("../Data/Air Travel.csv")

In [5]:
travel_rdd.collect()

['Dinesh Kapur,23',
 'Mahesh,34',
 'Manoj,19',
 'Remo,22',
 'John,55',
 'Ali,69',
 'Ram,45',
 'Gauri,40',
 'Seema,28',
 'Bhaskar,53']

In [6]:
pairs = travel_rdd.map(lambda row: row.split(',')) \
                    .map(lambda fields: (fields[0], int(fields[1])))

In [7]:
pairs.collect()

[('Dinesh Kapur', 23),
 ('Mahesh', 34),
 ('Manoj', 19),
 ('Remo', 22),
 ('John', 55),
 ('Ali', 69),
 ('Ram', 45),
 ('Gauri', 40),
 ('Seema', 28),
 ('Bhaskar', 53)]

In [8]:
grouped_pairs = pairs.map(lambda x, y: (x, ))

In [9]:
grouped_pairs = pairs.map(lambda row: (row[0], "Below 30" if row[1] < 30 
                                               else 'Below 50' if row[1] < 50 
                                               else 'Above 50'))

In [10]:
grouped_pairs.collect()

[('Dinesh Kapur', 'Below 30'),
 ('Mahesh', 'Below 50'),
 ('Manoj', 'Below 30'),
 ('Remo', 'Below 30'),
 ('John', 'Above 50'),
 ('Ali', 'Above 50'),
 ('Ram', 'Below 50'),
 ('Gauri', 'Below 50'),
 ('Seema', 'Below 30'),
 ('Bhaskar', 'Above 50')]

In [11]:
grouped_pairs.map(lambda row: (row[1], 1)) \
                .reduceByKey(lambda x, y: x + y) \
                .collect()

[('Below 30', 4), ('Below 50', 3), ('Above 50', 3)]